# Q-ErrorID: Haiqu error atlas

This notebook inspects the reproducible v0.5 outputs. Raw counts, independently readout-calibrated features, validation repeats, and held-out evaluation remain separate.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
RESULTS = ROOT / 'results' / 'haiqu'
ARTIFACTS = ROOT / 'artifacts' / 'haiqu'

## Experiment and device provenance

When `HAIQU_API_KEY` is available, the manifest contains experiment IDs and dashboard URLs. Local fallback runs are labelled and never represented as Haiqu executions.

In [ ]:
manifest = json.loads((ARTIFACTS / 'haiqu_experiments.json').read_text())
manifest

## Circuit analytics and raw reconstruction

In [ ]:
analytics = pd.read_csv(RESULTS / 'circuit_analytics.csv')
channels = pd.read_csv(RESULTS / 'reconstructed_channels.csv')
display(analytics.head())
display(channels)

## Independent readout calibration

Twenty raw basis-state circuits estimate four 1Q and three 2Q assignment matrices. Their regularized inverses are applied before the clean-feature Ridge model; the original counts remain in `feature_matrices.csv`.

In [ ]:
readout = pd.read_csv(RESULTS / 'readout_calibration.csv')
display(readout.groupby('calibration_key')[['condition_number', 'mean_assignment_fidelity', 'inverse_overhead_l1']].first())
json.loads((ARTIFACTS / 'readout_calibration.json').read_text())['validation_passed']

## Four-qubit local generator atlas

Node and edge values are model predictions. For fake/real backends they are validated empirically rather than treated as known parameters.

In [ ]:
display(Image(filename=str(RESULTS / 'device_error_atlas.png')))
pd.read_csv(RESULTS / 'device_error_atlas.csv')

## Haiqu mitigation modes

In [ ]:
mitigation = pd.read_csv(RESULTS / 'mitigation_comparison.csv')
display(mitigation.groupby(['mode', 'status']).size().rename('rows'))
display(Image(filename=str(RESULTS / 'mitigation_comparison.png')))

## Validated algorithm-level correction benchmark

The final track runs all four two-qubit Grover targets on each of the three selected edges. The response inverse uses `alpha`, `gamma`, and `kappa_down`, is fixed before Grover counts are observed, and adds no physical correction gates. Independent validation repeats select the generator inverse separately per edge; rejected edges retain readout-only mitigation. Final confidence intervals use held-out repeats.

In [ ]:
benchmark = pd.read_csv(RESULTS / 'final_benchmark.csv')
details = pd.read_csv(RESULTS / 'algorithm_benchmark_details.csv')
validation = pd.read_csv(RESULTS / 'correction_validation.csv')
seeds = pd.read_csv(RESULTS / 'benchmark_seed_summary.csv')
response = json.loads((ARTIFACTS / 'algorithm_response_models.json').read_text())
display(benchmark[['scenario', 'status', 'tvd_to_ideal', 'tvd_ci95_low', 'tvd_ci95_high', 'success_probability', 'response_condition_number']])
display(response['validation'])
display(seeds)
display(details.groupby(['scenario', 'edge_key'])[['tvd_to_ideal', 'success_probability']].mean(numeric_only=True))
display(pd.DataFrame({key: value['component_norms'] for key, value in response.get('models', {}).items()}).T)
display(Image(filename=str(RESULTS / 'final_benchmark.png')))

## Re-run the demo

From the repository root, export `HAIQU_API_KEY` and run:

```bash
python scripts/run_end_to_end_demo.py --device fake_fez --shots 4096 --validation-repeats 3 --evaluation-repeats 5
```